before this everytime run metadata table

In [0]:
from datetime import datetime
from pyspark.sql.functions import *

In [0]:
# Step 1: Read credentials from secret scope
username = dbutils.secrets.get(scope="ecommerce_scope",key="azureSQL-username")
password = dbutils.secrets.get(scope="ecommerce_scope",key="azureSQL-password")

# Step 2: JDBC connection details
jdbc_hostname = "azuresqlserverkviswan8.database.windows.net"
jdbc_port = 1433
jdbc_database = "ecommerce-data-pipeline-db"

jdbc_url = f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};database={jdbc_database}"

silver_transformations_rules = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "metadata_silver_config")  # schema.table
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

watermark_metadata = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "watermark_metadata")  # schema.table
    .option("user", username)
    .option("password", password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)


In [0]:
metadata_df = silver_transformations_rules.join(watermark_metadata,on="table_name",how="inner")
display(metadata_df)

In [0]:
# iterating each table

for row in metadata_df.toLocalIterator():
    table_name = row["table_name"]
    bronze_path = row["bronze_path"]
    silver_table = row["silver_table"]
    file_format = row["file_format"]
    primary_key = row["primary_key"]
    watermark_column = row["watermark_column"]
    is_active = row["is_active"]
    rules = row["rules"]
    silver_watermark_column = row["silver_watermark_column"]
    silver_last_processed = row["silver_last_processed"]

    folders = dbutils.fs.ls(bronze_path)
    
    current_date_is = datetime.today()
    valid_paths = []

    for folder in folders:
        # path = /bronzelayer/sales/customers/2026-07-22/
        # name = 2026-07-22/
        # folder.name has a trailing slash / : "2026-07-22/"
        # What .strip("/") does
        # Removes / from start & end
        # "2026-07-22/"  →  "2026-07-22"
        folder_name = folder.name.strip("/")
        
        try:
            # folder.name = "2026-07-22/"
            #         ↓ strip("/")
            # folder_name = "2026-07-22"
            #         ↓ strptime()
            # folder_date = datetime(2026, 7, 22)
            #         ↓
            # usable for comparisons
            folder_date = datetime.strptime(folder_name, "%Y-%m-%d")
            if folder_date > silver_last_processed and folder_date <= current_date_is:
                valid_paths.append(folder.path)
        except:
            print(f"Skipping invalid folder: {folder_name}")
    
    if len(valid_paths) == 0:
        print("There is no new Data")
    else:
        print(valid_paths)
